# Network Resilience and Traffic Flow Simulation

This notebook evaluates traffic flow, lane conflation, and network resilience using a Cellular Automata (CA) approach based on the Nagel-Schreckenberg model.

The core simulation logic is imported from `src/traffic_simulation.py` to maintain a clean analysis environment.

In [ ]:
# System and Environment Setup
import sys
import os
import warnings

# Add the root directory to the path so we can import our custom module
sys.path.append(os.path.abspath('..'))
from src.traffic_simulation import run_merge_simulation

# Data Handling and Visualization
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Configure visual style and suppress division warnings from empty arrays
sns.set_theme(style="whitegrid")
warnings.filterwarnings("ignore", category=RuntimeWarning)

## Scenario 1: Street Conflation & Priority Rules

We extend the linear street model such that two lanes ("green" and "red") conflate into a single lane in the middle of the road segment. We test two priority rules:
* **Rule 1:** The car closest to the conflation point goes first.
* **Rule 2:** The green lane has absolute priority over the red lane.

In [ ]:
# Base Model Parameters
Ncells = 100
merge_start = 40
merge_end = 60
vmax = 5
pb = 0.2
nsteps = 50

# Configurations to test: (Red Cars, Green Cars)
car_configs = [(10, 10), (15, 5), (5, 15)]
rules = [1, 2]
results = {}

for rule in rules:
    for NR, NG in car_configs:
        initial_zone = 30
        np.random.seed(42) # Set seed for reproducibility

        # Generate random initial positions
        red_pos = np.random.choice(range(initial_zone), size=NR, replace=False)
        green_pos = np.random.choice(range(initial_zone), size=NG, replace=False)

        # Create position dictionaries
        red_cars = {f'R{j}': pos for j, pos in enumerate(red_pos)}
        green_cars = {f'G{j}': pos for j, pos in enumerate(green_pos)}
        initial_positions = {**red_cars, **green_cars}

        # Run the simulation using the imported module
        simulation_df = run_merge_simulation(
            nsteps=nsteps,
            positions=initial_positions,
            pb=pb,
            vmax=vmax,
            ncells=Ncells,
            merge_start=merge_start,
            merge_end=merge_end,
            rule=rule
        )

        # Store results for plotting
        results[f'Rule{rule}_Red{NR}_Green{NG}'] = simulation_df

print("Scenario 1 Simulations Complete.")

In [ ]:
# Plotting Average Velocity Over Time
plt.figure(figsize=(14, 6))

for key, df in results.items():
    velocities = []

    for t in range(1, len(df)):
        prev = df.iloc[t - 1, :]
        curr = df.iloc[t, :]

        # Calculate velocity (difference in position)
        v = np.nanmean(curr.values - prev.values)
        velocities.append(v)

    velocities = np.nan_to_num(velocities, nan=0.0)

    if len(velocities) > 0:
        plt.plot(range(1, len(df)), velocities, label=str(key), linewidth=1.5)

plt.xlabel("Time Step", fontsize=12)
plt.ylabel("Average Velocity", fontsize=12)
plt.title("Average Velocity Over Time for Different Merge Rules and Lane Ratios", fontsize=14)
plt.legend(title="Configuration", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

In [ ]:
# Plotting Throughput (Cars Passing Cell 70 per 10 Timesteps)
plt.figure(figsize=(14, 6))
target_cell = 70
interval = 10

for key, df in results.items():
    throughput = []
    time_intervals = range(interval, nsteps + 1, interval)

    for end_t in time_intervals:
        start_t = end_t - interval
        passed = 0

        # Check each car to see if it crossed the target cell during this interval
        for car in df.columns:
            pos_start = df.iloc[start_t][car] if start_t < len(df) and pd.notna(df.iloc[start_t][car]) else 0
            pos_end = df.iloc[end_t - 1][car] if end_t - 1 < len(df) and pd.notna(df.iloc[end_t - 1][car]) else Ncells

            if pos_start < target_cell and pos_end >= target_cell:
                passed += 1

        throughput.append(passed)

    plt.plot(time_intervals, throughput, marker='o', label=str(key), linewidth=1.5)

plt.xlabel("Time Step Interval (End)", fontsize=12)
plt.ylabel(f"Cars Passing Cell {target_cell}", fontsize=12)
plt.title(f"Throughput: Cars Passing Cell {target_cell} per {interval} Timesteps", fontsize=14)
plt.legend(title="Configuration", bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

## Scenario 2: Traffic Jam & Resilience Analysis

To inspect system resilience, we induce a traffic jam on the conflated road by strictly blocking a specific cell close to the end of the segment for 10 timesteps. We then observe how quickly the traffic flow recovers once the blockage is lifted.

In [ ]:
# Traffic Jam Parameters
nsteps_jam = 80
jam_pos = 50
jam_start = 15
jam_end = 25

# Generate a balanced baseline configuration for the test
NR, NG = 10, 10
np.random.seed(42)
red_pos = np.random.choice(range(30), size=NR, replace=False)
green_pos = np.random.choice(range(30), size=NG, replace=False)
baseline_positions = {**{f'R{j}': pos for j, pos in enumerate(red_pos)},
                      **{f'G{j}': pos for j, pos in enumerate(green_pos)}}

# 1. Run Baseline (No Jam)
df_no_jam = run_merge_simulation(nsteps_jam, baseline_positions, pb=pb, vmax=vmax, ncells=Ncells, merge_start=merge_start, merge_end=merge_end, rule=1)

# 2. Run Jam Scenario
df_jam = run_merge_simulation(nsteps_jam, baseline_positions, pb=pb, vmax=vmax, ncells=Ncells, merge_start=merge_start, merge_end=merge_end, rule=1, jam_pos=jam_pos, jam_start=jam_start, jam_end=jam_end)

# Calculate velocities for both scenarios
vel_no_jam, vel_jam = [], []
for t in range(1, len(df_no_jam)):
    vel_no_jam.append(np.nanmean(df_no_jam.iloc[t, :].values - df_no_jam.iloc[t - 1, :].values))
    vel_jam.append(np.nanmean(df_jam.iloc[t, :].values - df_jam.iloc[t - 1, :].values))

# Plot the Resilience Comparison
plt.figure(figsize=(14, 6))
plt.plot(range(1, len(df_no_jam)), np.nan_to_num(vel_no_jam), label="Without Jam", color="blue", linewidth=1.5)
plt.plot(range(1, len(df_jam)), np.nan_to_num(vel_jam), label="With Jam", color="red", linewidth=1.5)

# Highlight the jam period
plt.axvspan(jam_start, jam_end, color='red', alpha=0.2, label='Jam Period')

plt.xlabel("Time Step", fontsize=12)
plt.ylabel("Average Velocity", fontsize=12)
plt.title("System Resilience: Average Velocity Recovery After Traffic Jam", fontsize=14)
plt.legend()
plt.tight_layout()
plt.show()